
# Football Roboflow Tracking v12

Amaç:
- Roboflow API ile futbol modeli / workflow üzerinden detection almak
- Oyuncu, kaleci, hakem ve topu ayrı takip etmek
- Oyuncuları forma rengine göre Team A / Team B olarak ayırmak
- Team A, Team B, Referee ve Ball için farklı renk/marker çizmek
- En iyi confidence ayarını örnek frame'lerde otomatik denemek
- Video ve CSV çıktısı üretmek

> Not: Roboflow Universe sayfasında `football-player-jrjtj` projesi dataset olarak görünüyor. Eğer `football-player-jrjtj/1` model ID çalışmazsa sayfadaki Code/Deploy kısmından workflow bilgilerini girip `ROBOFLOW_MODE = "workflow"` yap.


In [ ]:

# ============================================================
# CELL 1 - Kurulum
# ============================================================
!pip -q install inference-sdk supervision opencv-python-headless pandas numpy tqdm scikit-learn


In [ ]:

# ============================================================
# CELL 2 - Ayarlar
# ============================================================
import os
from pathlib import Path

# Colab/Kaggle video path
VIDEO_PATH = "/content/input_video.mp4"
OUTPUT_DIR = "/content/football_v12_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Roboflow ayarları
# -----------------------------
ROBOFLOW_API_KEY = ""  # kendi API key'ini yaz
ROBOFLOW_API_URL = "https://serverless.roboflow.com"

# auto: önce model_id dener, olmazsa workflow bilgileri girildiyse workflow dener
# model: sadece CLIENT.infer(..., model_id=...) dener
# workflow: client.run_workflow(...) dener
ROBOFLOW_MODE = "auto"

# Kullanmak istediğin proje. Bu model ID çalışmayabilir çünkü sayfada Models 0 görünüyor.
ROBOFLOW_MODEL_ID = "football-player-jrjtj/1"

# Eğer Universe Code sekmesi workflow veriyorsa doldur:
ROBOFLOW_WORKSPACE = "minaehyeon"
ROBOFLOW_WORKFLOW_ID = ""  # örnek: "detect-ball-player-referee" gibi sayfadan kopyala
ROBOFLOW_CLASSES = "ball, player, referee, goalkeeper"

# Eski çalışan modelin varsa burada fallback açabilirsin.
# Boş bırakırsan sadece yukarıdaki modeli/workflow'u kullanır.
FALLBACK_MODEL_IDS = [
    # "football-players-detection-3zvbc/11",
]

# -----------------------------
# Otomatik threshold tuning
# -----------------------------
RUN_THRESHOLD_TUNING = True
TUNING_SAMPLE_FRAMES = 35        # API çağrısı sayısı. Maliyet için 20-40 arası iyi.
TUNING_MIN_GAP_FRAMES = 10

PLAYER_THRESHOLDS = [0.12, 0.15, 0.18, 0.20, 0.25]
GOALKEEPER_THRESHOLDS = [0.10, 0.15, 0.20, 0.25]
REFEREE_THRESHOLDS = [0.12, 0.15, 0.20, 0.25]
BALL_THRESHOLDS = [0.04, 0.06, 0.08, 0.10, 0.12]

# Eğer tuning kapalıysa kullanılacak değerler
DEFAULT_THRESHOLDS = {
    "player": 0.18,
    "goalkeeper": 0.15,
    "referee": 0.15,
    "ball": 0.06,
}

# Beklenen sayılar. Bunlar tuning skorunda kullanılır, kesin ground truth değildir.
EXPECTED_PLAYERS_MIN = 16
EXPECTED_PLAYERS_MAX = 24
EXPECTED_REFEREES_MIN = 0
EXPECTED_REFEREES_MAX = 4
EXPECTED_BALL_MIN = 0
EXPECTED_BALL_MAX = 2

# -----------------------------
# Tracking ayarları
# -----------------------------
DETECT_EVERY_N_FRAMES = 1        # En iyi kalite için 1. Hız için 2/3 yapılabilir.
PROCESS_MAX_FRAMES = None       # Test için 300 yaz, tüm video için None

# ByteTrack ayarları. Tracking kopuyorsa lost buffer artır.
TRACK_ACTIVATION_THRESHOLD = 0.10
LOST_TRACK_BUFFER = 90
MIN_MATCHING_THRESHOLD = 0.70

# Stable ID ayarları
PLAYER_STABLE_MAX_DISTANCE = 160
REF_STABLE_MAX_DISTANCE = 140
BALL_STABLE_MAX_DISTANCE = 220
STABLE_MAX_MISSING_FRAMES = 120

# Takım ayrımı
TEAM_CALIBRATION_FRAMES = 80     # İlk kaç frame'den forma rengi toplansın
TEAM_MIN_SAMPLES = 30
TEAM_VOTE_MEMORY = 20            # track için son kaç team tahmini oylansın
JERSEY_CROP_DEBUG = False

# Çizim
DRAW_BOX = False
DRAW_ELLIPSE = True
SHOW_ID = True
SHOW_CLASS = True
SHOW_CONF = False
SHOW_TEAM = True
SHOW_BALL_TRAIL = True
BALL_TRAIL_LENGTH = 20

# Renkler BGR formatında
COLOR_TEAM_A = (255, 80, 80)       # mavi ton
COLOR_TEAM_B = (80, 80, 255)       # kırmızı ton
COLOR_REFEREE = (0, 255, 255)      # sarı
COLOR_GOALKEEPER = (255, 0, 255)   # mor
COLOR_BALL = (255, 255, 255)       # beyaz
COLOR_UNKNOWN = (200, 200, 200)

print("Ayarlar hazır.")


In [ ]:

# ============================================================
# CELL 3 - Importlar
# ============================================================
import cv2
import math
import time
import json
import itertools
import tempfile
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict, deque, Counter

import supervision as sv
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from inference_sdk import InferenceHTTPClient


def check_video(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Video bulunamadı: {path}. VIDEO_PATH değerini düzelt.")
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f"Video: {path}")
    print(f"FPS: {fps:.2f} | Size: {w}x{h} | Frames: {total}")
    return fps, w, h, total


def frame_to_temp_jpg(frame):
    tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
    path = tmp.name
    tmp.close()
    cv2.imwrite(path, frame)
    return path

print("Importlar hazır.")


In [ ]:

# ============================================================
# CELL 4 - Roboflow client + prediction parser
# ============================================================
if not ROBOFLOW_API_KEY:
    raise ValueError("ROBOFLOW_API_KEY boş. Roboflow API key'i CELL 2 içine yazmalısın.")

CLIENT = InferenceHTTPClient(api_url=ROBOFLOW_API_URL, api_key=ROBOFLOW_API_KEY)

CANONICAL_CLASSES = {"player", "goalkeeper", "referee", "ball"}

CLASS_ALIASES = {
    "players": "player",
    "football-player": "player",
    "football_player": "player",
    "person": "player",
    "goalkeepers": "goalkeeper",
    "goal keeper": "goalkeeper",
    "ref": "referee",
    "referees": "referee",
    "soccer-ball": "ball",
    "football": "ball",
    "sports ball": "ball",
}


def normalize_class_name(name):
    s = str(name).strip().lower().replace("_", "-")
    s = CLASS_ALIASES.get(s, s)
    if s not in CANONICAL_CLASSES:
        # Bazı Roboflow çıktıları class adı yerine class_id verebilir.
        return s
    return s


def extract_prediction_lists(obj):
    """Roboflow infer veya workflow sonucu içinden prediction listelerini bulur."""
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ["predictions", "detections"] and isinstance(v, list):
                if len(v) == 0 or isinstance(v[0], dict):
                    found.append(v)
            found.extend(extract_prediction_lists(v))
    elif isinstance(obj, list):
        # Direkt prediction listesi olabilir
        if len(obj) > 0 and isinstance(obj[0], dict) and any(key in obj[0] for key in ["x", "bbox", "points"]):
            found.append(obj)
        for item in obj:
            found.extend(extract_prediction_lists(item))
    return found


def prediction_to_xyxy(p):
    """Farklı Roboflow prediction formatlarını xyxy'ye çevirir."""
    if all(k in p for k in ["x", "y", "width", "height"]):
        x = float(p["x"]); y = float(p["y"]); w = float(p["width"]); h = float(p["height"])
        return [x - w/2, y - h/2, x + w/2, y + h/2]

    if "bbox" in p:
        b = p["bbox"]
        if isinstance(b, dict):
            if all(k in b for k in ["x", "y", "width", "height"]):
                x = float(b["x"]); y = float(b["y"]); w = float(b["width"]); h = float(b["height"])
                return [x - w/2, y - h/2, x + w/2, y + h/2]
            if all(k in b for k in ["x1", "y1", "x2", "y2"]):
                return [float(b["x1"]), float(b["y1"]), float(b["x2"]), float(b["y2"])]
        if isinstance(b, (list, tuple)) and len(b) == 4:
            return [float(b[0]), float(b[1]), float(b[2]), float(b[3])]

    if all(k in p for k in ["x1", "y1", "x2", "y2"]):
        return [float(p["x1"]), float(p["y1"]), float(p["x2"]), float(p["y2"])]

    return None


def parse_roboflow_result(result):
    pred_lists = extract_prediction_lists(result)
    preds = []
    for lst in pred_lists:
        preds.extend(lst)

    parsed = []
    seen = set()
    for p in preds:
        xyxy = prediction_to_xyxy(p)
        if xyxy is None:
            continue

        raw_cls = p.get("class", p.get("class_name", p.get("label", p.get("name", "unknown"))))
        cls = normalize_class_name(raw_cls)
        if cls not in CANONICAL_CLASSES:
            continue

        conf = float(p.get("confidence", p.get("score", p.get("conf", 0.0))))
        key = (round(xyxy[0], 1), round(xyxy[1], 1), round(xyxy[2], 1), round(xyxy[3], 1), cls, round(conf, 3))
        if key in seen:
            continue
        seen.add(key)

        parsed.append({
            "class": cls,
            "confidence": conf,
            "xyxy": xyxy,
            "raw": p,
        })
    return parsed


def call_roboflow_model(image_path, model_id):
    return CLIENT.infer(image_path, model_id=model_id)


def call_roboflow_workflow(image_path):
    if not ROBOFLOW_WORKFLOW_ID:
        raise ValueError("ROBOFLOW_WORKFLOW_ID boş. Workflow kullanmak için Code sekmesinden workflow_id gir.")
    return CLIENT.run_workflow(
        workspace_name=ROBOFLOW_WORKSPACE,
        workflow_id=ROBOFLOW_WORKFLOW_ID,
        images={"image": image_path},
        parameters={"classes": ROBOFLOW_CLASSES},
        use_cache=True,
    )


def roboflow_predict_raw(frame):
    """Frame için raw parsed predictions döndürür."""
    image_path = frame_to_temp_jpg(frame)
    errors = []
    try:
        modes_to_try = []
        if ROBOFLOW_MODE == "model":
            modes_to_try = [("model", ROBOFLOW_MODEL_ID)] + [("model", m) for m in FALLBACK_MODEL_IDS]
        elif ROBOFLOW_MODE == "workflow":
            modes_to_try = [("workflow", None)]
        else:
            modes_to_try = [("model", ROBOFLOW_MODEL_ID)] + [("model", m) for m in FALLBACK_MODEL_IDS]
            if ROBOFLOW_WORKFLOW_ID:
                modes_to_try.append(("workflow", None))

        for mode, mid in modes_to_try:
            try:
                if mode == "model":
                    result = call_roboflow_model(image_path, mid)
                else:
                    result = call_roboflow_workflow(image_path)
                parsed = parse_roboflow_result(result)
                return parsed, {"mode": mode, "model_id": mid, "raw_result_type": type(result).__name__}
            except Exception as e:
                errors.append(f"{mode}:{mid} -> {repr(e)}")

        raise RuntimeError("Roboflow çağrısı başarısız:
" + "
".join(errors))
    finally:
        try:
            os.remove(image_path)
        except Exception:
            pass

print("Roboflow parser hazır.")


In [ ]:

# ============================================================
# CELL 5 - Threshold filtreleme ve supervision Detections dönüşümü
# ============================================================
CLASS_TO_ID = {
    "player": 0,
    "goalkeeper": 1,
    "referee": 2,
    "ball": 3,
}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}


def filter_predictions(preds, thresholds):
    out = []
    for p in preds:
        cls = p["class"]
        conf = float(p["confidence"])
        th = thresholds.get(cls, 0.25)
        if conf >= th:
            out.append(p)
    return out


def predictions_to_detections(preds):
    if len(preds) == 0:
        return sv.Detections.empty(), []
    xyxy = np.array([p["xyxy"] for p in preds], dtype=np.float32)
    conf = np.array([p["confidence"] for p in preds], dtype=np.float32)
    cls_ids = np.array([CLASS_TO_ID[p["class"]] for p in preds], dtype=int)
    class_names = [p["class"] for p in preds]
    return sv.Detections(xyxy=xyxy, confidence=conf, class_id=cls_ids), class_names


def split_detections_by_class(detections):
    if len(detections) == 0:
        return {c: sv.Detections.empty() for c in CANONICAL_CLASSES}
    result = {}
    for cname, cid in CLASS_TO_ID.items():
        mask = detections.class_id == cid
        result[cname] = detections[mask]
    return result

print("Detection filtreleme hazır.")


In [ ]:

# ============================================================
# CELL 6 - Örnek frame'lerde threshold tuning
# ============================================================
def get_sample_frame_indices(total_frames, n_samples, min_gap=10):
    if total_frames <= 0:
        return []
    n = min(n_samples, max(1, total_frames // max(1, min_gap)))
    idxs = np.linspace(0, total_frames - 1, n, dtype=int).tolist()
    # tekrarları kaldır
    clean = []
    last = -10**9
    for i in idxs:
        if i - last >= min_gap:
            clean.append(int(i))
            last = i
    return clean


def read_specific_frames(video_path, frame_indices):
    cap = cv2.VideoCapture(video_path)
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            frames.append((idx, frame))
    cap.release()
    return frames


def count_by_class(preds, thresholds):
    filtered = filter_predictions(preds, thresholds)
    cnt = Counter([p["class"] for p in filtered])
    return {
        "player": cnt.get("player", 0) + cnt.get("goalkeeper", 0),
        "field_player": cnt.get("player", 0),
        "goalkeeper": cnt.get("goalkeeper", 0),
        "referee": cnt.get("referee", 0),
        "ball": cnt.get("ball", 0),
        "total": len(filtered),
    }


def range_score(value, low, high):
    if low <= value <= high:
        return 1.0
    if value < low:
        return max(0.0, 1.0 - (low - value) / max(1.0, low))
    return max(0.0, 1.0 - (value - high) / max(1.0, high))


def tune_thresholds(video_path):
    fps, w, h, total = check_video(video_path)
    sample_indices = get_sample_frame_indices(total, TUNING_SAMPLE_FRAMES, TUNING_MIN_GAP_FRAMES)
    sample_frames = read_specific_frames(video_path, sample_indices)
    print(f"Tuning sample frame sayısı: {len(sample_frames)}")

    raw_by_frame = []
    rf_info = None
    for frame_idx, frame in tqdm(sample_frames, desc="Roboflow raw detections"):
        preds, info = roboflow_predict_raw(frame)
        rf_info = info
        raw_by_frame.append((frame_idx, preds))

    print("Roboflow çağrı bilgisi:", rf_info)

    rows = []
    best = None
    best_score = -1

    grid = itertools.product(PLAYER_THRESHOLDS, GOALKEEPER_THRESHOLDS, REFEREE_THRESHOLDS, BALL_THRESHOLDS)
    for pth, gkth, rth, bth in grid:
        thresholds = {"player": pth, "goalkeeper": gkth, "referee": rth, "ball": bth}
        counts = [count_by_class(preds, thresholds) for _, preds in raw_by_frame]
        if not counts:
            continue
        dfc = pd.DataFrame(counts)

        mean_players = dfc["player"].mean()
        mean_refs = dfc["referee"].mean()
        mean_balls = dfc["ball"].mean()
        std_players = dfc["player"].std() if len(dfc) > 1 else 0

        # Skor: Oyuncu sayısı makul olsun, ref/top aşırı çoğalmasın, frame'ler arası çok dalgalanmasın.
        player_score = range_score(mean_players, EXPECTED_PLAYERS_MIN, EXPECTED_PLAYERS_MAX)
        ref_score = range_score(mean_refs, EXPECTED_REFEREES_MIN, EXPECTED_REFEREES_MAX)
        ball_score = range_score(mean_balls, EXPECTED_BALL_MIN, EXPECTED_BALL_MAX)
        stability_score = max(0.0, 1.0 - std_players / max(1.0, mean_players))

        score = 0.50 * player_score + 0.15 * ref_score + 0.15 * ball_score + 0.20 * stability_score

        row = {
            "player_th": pth, "goalkeeper_th": gkth, "referee_th": rth, "ball_th": bth,
            "score": score,
            "mean_players": mean_players,
            "mean_field_players": dfc["field_player"].mean(),
            "mean_goalkeepers": dfc["goalkeeper"].mean(),
            "mean_referees": mean_refs,
            "mean_balls": mean_balls,
            "std_players": std_players,
        }
        rows.append(row)
        if score > best_score:
            best_score = score
            best = thresholds.copy()

    tuning_df = pd.DataFrame(rows).sort_values("score", ascending=False)
    tuning_path = os.path.join(OUTPUT_DIR, "threshold_tuning_results.csv")
    tuning_df.to_csv(tuning_path, index=False)

    print("
En iyi threshold ayarı:", best)
    print("Tuning CSV:", tuning_path)
    display(tuning_df.head(15))
    return best, tuning_df

if RUN_THRESHOLD_TUNING:
    BEST_THRESHOLDS, tuning_df = tune_thresholds(VIDEO_PATH)
else:
    BEST_THRESHOLDS = DEFAULT_THRESHOLDS.copy()

print("Kullanılacak thresholdlar:", BEST_THRESHOLDS)


In [ ]:

# ============================================================
# CELL 7 - Takım renk modeli
# ============================================================
def clip_xyxy(xyxy, frame_shape):
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = map(int, xyxy)
    x1 = max(0, min(w - 1, x1)); x2 = max(0, min(w - 1, x2))
    y1 = max(0, min(h - 1, y1)); y2 = max(0, min(h - 1, y2))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def extract_jersey_feature(frame, xyxy):
    box = clip_xyxy(xyxy, frame.shape)
    if box is None:
        return None
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    if bw < 8 or bh < 20:
        return None

    # Gövde/forma bölgesi: üst-orta kısım
    cx1 = int(x1 + 0.20 * bw)
    cx2 = int(x1 + 0.80 * bw)
    cy1 = int(y1 + 0.18 * bh)
    cy2 = int(y1 + 0.58 * bh)
    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)

    # Çim yeşilini ve çok karanlık/aşırı parlak pikseli azalt
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    non_grass = ~((h > 35) & (h < 90) & (s > 45) & (v > 40))
    valid = non_grass & (v > 35) & (v < 245) & (s > 20)

    if valid.sum() < 20:
        valid = (v > 30) & (v < 250)
    if valid.sum() < 20:
        return None

    # Robust median renk özellikleri
    hsv_med = np.median(hsv[valid], axis=0)
    lab_med = np.median(lab[valid], axis=0)
    bgr_med = np.median(crop[valid], axis=0)

    # H circular olduğu için sin/cos ile temsil ediyoruz
    hue = hsv_med[0] * 2 * np.pi / 180.0
    feat = np.array([
        np.sin(hue), np.cos(hue),
        hsv_med[1] / 255.0, hsv_med[2] / 255.0,
        lab_med[1] / 255.0, lab_med[2] / 255.0,
        bgr_med[0] / 255.0, bgr_med[1] / 255.0, bgr_med[2] / 255.0,
    ], dtype=float)
    return feat


class TeamColorClassifier:
    def __init__(self, min_samples=30):
        self.min_samples = min_samples
        self.samples = []
        self.model = None
        self.scaler = None
        self.ready = False

    def add_sample(self, feature):
        if feature is not None and np.all(np.isfinite(feature)):
            self.samples.append(feature)

    def fit(self):
        if len(self.samples) < self.min_samples:
            print(f"Takım modeli için örnek az: {len(self.samples)}/{self.min_samples}")
            return False
        X = np.vstack(self.samples)
        self.scaler = StandardScaler()
        Xs = self.scaler.fit_transform(X)
        self.model = KMeans(n_clusters=2, random_state=42, n_init=20)
        labels = self.model.fit_predict(Xs)
        self.ready = True
        counts = Counter(labels)
        print("Takım modeli hazır. Cluster dağılımı:", dict(counts))
        return True

    def predict(self, feature):
        if not self.ready or feature is None:
            return "Unknown"
        Xs = self.scaler.transform(feature.reshape(1, -1))
        label = int(self.model.predict(Xs)[0])
        return "Team A" if label == 0 else "Team B"

print("TeamColorClassifier hazır.")


In [ ]:

# ============================================================
# CELL 8 - Stable ID + team voting
# ============================================================
class StableIDManager:
    def __init__(self, prefix, max_center_distance=150, max_missing_frames=120):
        self.prefix = prefix
        self.max_center_distance = max_center_distance
        self.max_missing_frames = max_missing_frames
        self.next_id = 1
        self.tracker_to_stable = {}
        self.state = {}

    def _center(self, xyxy):
        x1, y1, x2, y2 = xyxy
        return np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=float)

    def update(self, tracker_id, xyxy, frame_idx):
        center = self._center(xyxy)
        if tracker_id is not None and tracker_id >= 0 and tracker_id in self.tracker_to_stable:
            sid = self.tracker_to_stable[tracker_id]
            self.state[sid] = {"center": center, "frame_idx": frame_idx, "hits": self.state.get(sid, {}).get("hits", 0) + 1}
            return sid

        best_sid, best_dist = None, float("inf")
        for sid, st in self.state.items():
            age = frame_idx - st.get("frame_idx", -10**9)
            if age < 0 or age > self.max_missing_frames:
                continue
            dist = float(np.linalg.norm(center - st["center"]))
            if dist < best_dist and dist <= self.max_center_distance:
                best_sid, best_dist = sid, dist

        if best_sid is None:
            best_sid = f"{self.prefix}{self.next_id}"
            self.next_id += 1

        if tracker_id is not None and tracker_id >= 0:
            self.tracker_to_stable[tracker_id] = best_sid
        self.state[best_sid] = {"center": center, "frame_idx": frame_idx, "hits": self.state.get(best_sid, {}).get("hits", 0) + 1}
        return best_sid

    def cleanup(self, frame_idx):
        dead = [sid for sid, st in self.state.items() if frame_idx - st.get("frame_idx", 0) > self.max_missing_frames * 3]
        for sid in dead:
            self.state.pop(sid, None)


class TeamVoteMemory:
    def __init__(self, memory=20):
        self.memory = memory
        self.votes = defaultdict(lambda: deque(maxlen=memory))

    def update(self, stable_id, team_label):
        if team_label in ["Team A", "Team B"]:
            self.votes[stable_id].append(team_label)
        if len(self.votes[stable_id]) == 0:
            return "Unknown"
        return Counter(self.votes[stable_id]).most_common(1)[0][0]

print("Stable ID ve team voting hazır.")


In [ ]:

# ============================================================
# CELL 9 - Çizim fonksiyonları
# ============================================================
def color_for_object(class_name, team_label="Unknown"):
    if class_name == "referee":
        return COLOR_REFEREE
    if class_name == "ball":
        return COLOR_BALL
    if class_name == "goalkeeper":
        # İstersen kaleciyi takım rengine göre de boyayabiliriz; şimdilik ayırt edilsin diye mor.
        return COLOR_GOALKEEPER
    if team_label == "Team A":
        return COLOR_TEAM_A
    if team_label == "Team B":
        return COLOR_TEAM_B
    return COLOR_UNKNOWN


def draw_text_with_bg(frame, text, org, color=(255,255,255), font_scale=0.55, thickness=1):
    x, y = int(org[0]), int(org[1])
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), base = cv2.getTextSize(text, font, font_scale, thickness)
    cv2.rectangle(frame, (x, y - th - base - 5), (x + tw + 8, y + base + 3), (0, 0, 0), -1)
    cv2.putText(frame, text, (x + 4, y - 3), font, font_scale, color, thickness, cv2.LINE_AA)


def draw_ellipse_under_player(frame, xyxy, color, thickness=3):
    x1, y1, x2, y2 = map(int, xyxy)
    cx = int((x1 + x2) / 2)
    bottom = int(y2)
    bw = max(10, x2 - x1)
    axes = (max(10, int(bw * 0.48)), max(5, int(bw * 0.16)))
    cv2.ellipse(frame, (cx, bottom), axes, 0, 0, 360, color, thickness)


def draw_detection(frame, xyxy, class_name, stable_id, team_label, confidence=None):
    color = color_for_object(class_name, team_label)
    x1, y1, x2, y2 = map(int, xyxy)

    if class_name == "ball":
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        r = max(5, int(max(x2 - x1, y2 - y1) / 2))
        cv2.circle(frame, (cx, cy), r, color, 2)
        cv2.circle(frame, (cx, cy), 2, color, -1)
    else:
        if DRAW_ELLIPSE:
            draw_ellipse_under_player(frame, xyxy, color, thickness=3)
        if DRAW_BOX:
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    label = []
    if SHOW_CLASS:
        if class_name == "player" and SHOW_TEAM:
            label.append(team_label)
        else:
            label.append(class_name)
    if SHOW_ID:
        label.append(str(stable_id))
    if SHOW_CONF and confidence is not None:
        label.append(f"{confidence:.2f}")
    if label:
        draw_text_with_bg(frame, " | ".join(label), (x1, max(20, y1 - 6)), color=color)


def draw_ball_trail(frame, points):
    if not SHOW_BALL_TRAIL or len(points) < 2:
        return
    pts = list(points)
    for i in range(1, len(pts)):
        thickness = max(1, int(4 * i / len(pts)))
        cv2.line(frame, pts[i-1], pts[i], COLOR_BALL, thickness)

print("Çizim fonksiyonları hazır.")


In [ ]:

# ============================================================
# CELL 10 - Takım kalibrasyonu
# ============================================================
def calibrate_team_classifier(video_path, thresholds):
    fps, w, h, total = check_video(video_path)
    sample_count = min(TEAM_CALIBRATION_FRAMES, total)
    indices = get_sample_frame_indices(total, sample_count, max(1, total // max(1, sample_count)))
    if len(indices) > TEAM_CALIBRATION_FRAMES:
        indices = indices[:TEAM_CALIBRATION_FRAMES]

    classifier = TeamColorClassifier(min_samples=TEAM_MIN_SAMPLES)
    frames = read_specific_frames(video_path, indices)

    for frame_idx, frame in tqdm(frames, desc="Team calibration"):
        preds, _ = roboflow_predict_raw(frame)
        filtered = filter_predictions(preds, thresholds)
        for p in filtered:
            # Sadece saha oyuncuları üzerinden takım rengi öğrenmek daha doğru.
            if p["class"] != "player":
                continue
            feat = extract_jersey_feature(frame, p["xyxy"])
            classifier.add_sample(feat)

    classifier.fit()
    return classifier

team_classifier = calibrate_team_classifier(VIDEO_PATH, BEST_THRESHOLDS)


In [ ]:

# ============================================================
# CELL 11 - Ana video işleme
# ============================================================
def make_tracker(fps):
    return sv.ByteTrack(
        track_activation_threshold=TRACK_ACTIVATION_THRESHOLD,
        lost_track_buffer=LOST_TRACK_BUFFER,
        minimum_matching_threshold=MIN_MATCHING_THRESHOLD,
        frame_rate=max(1, int(round(fps))),
    )


def get_tracker_ids(detections):
    if len(detections) == 0 or detections.tracker_id is None:
        return []
    return [int(x) for x in detections.tracker_id]


def merge_detections(items):
    valid = [d for d in items if len(d) > 0]
    if not valid:
        return sv.Detections.empty()
    if len(valid) == 1:
        return valid[0]
    return sv.Detections.merge(valid)


def process_video(video_path, thresholds):
    fps, width, height, total = check_video(video_path)
    total_to_process = total if PROCESS_MAX_FRAMES is None else min(total, PROCESS_MAX_FRAMES)

    player_tracker = make_tracker(fps)
    referee_tracker = make_tracker(fps)
    ball_tracker = make_tracker(fps)

    player_sid = StableIDManager("P", PLAYER_STABLE_MAX_DISTANCE, STABLE_MAX_MISSING_FRAMES)
    referee_sid = StableIDManager("R", REF_STABLE_MAX_DISTANCE, STABLE_MAX_MISSING_FRAMES)
    ball_sid = StableIDManager("B", BALL_STABLE_MAX_DISTANCE, STABLE_MAX_MISSING_FRAMES)
    team_memory = TeamVoteMemory(TEAM_VOTE_MEMORY)
    ball_points = deque(maxlen=BALL_TRAIL_LENGTH)

    out_video = os.path.join(OUTPUT_DIR, "roboflow_v12_team_ref_ball_tracking.mp4")
    out_csv = os.path.join(OUTPUT_DIR, "roboflow_v12_tracking.csv")
    out_summary = os.path.join(OUTPUT_DIR, "roboflow_v12_summary.json")

    writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError("VideoWriter açılamadı.")

    cap = cv2.VideoCapture(video_path)
    rows = []
    stats = defaultdict(int)
    frame_idx = 0
    start = time.time()

    last_raw_preds = []

    pbar = tqdm(total=total_to_process, desc="Tracking video")
    while frame_idx < total_to_process:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % DETECT_EVERY_N_FRAMES == 0:
            raw_preds, _ = roboflow_predict_raw(frame)
            last_raw_preds = raw_preds
        else:
            raw_preds = last_raw_preds

        filtered_preds = filter_predictions(raw_preds, thresholds)
        detections, class_names = predictions_to_detections(filtered_preds)
        split = split_detections_by_class(detections)

        # player + goalkeeper birlikte player tracker'a girsin
        player_like = merge_detections([split["player"], split["goalkeeper"]])
        tracked_players = player_tracker.update_with_detections(player_like) if len(player_like) else sv.Detections.empty()
        tracked_refs = referee_tracker.update_with_detections(split["referee"]) if len(split["referee"]) else sv.Detections.empty()
        tracked_balls = ball_tracker.update_with_detections(split["ball"]) if len(split["ball"]) else sv.Detections.empty()

        annotated = frame.copy()

        # Oyuncu / kaleci çizimi
        for i in range(len(tracked_players)):
            xyxy = tracked_players.xyxy[i]
            conf = float(tracked_players.confidence[i]) if tracked_players.confidence is not None else None
            cid = int(tracked_players.class_id[i]) if tracked_players.class_id is not None else CLASS_TO_ID["player"]
            cls = ID_TO_CLASS.get(cid, "player")
            tid = int(tracked_players.tracker_id[i]) if tracked_players.tracker_id is not None else -1
            sid = player_sid.update(tid, xyxy, frame_idx)

            team_pred = "Unknown"
            if cls == "player":
                feat = extract_jersey_feature(frame, xyxy)
                team_pred = team_classifier.predict(feat)
                team_label = team_memory.update(sid, team_pred)
            else:
                # Kaleci için forma rengi farklı olabileceği için class olarak kaleci çiziyoruz.
                team_label = "Goalkeeper"

            draw_detection(annotated, xyxy, cls, sid, team_label, conf)

            rows.append({
                "frame": frame_idx,
                "time_sec": frame_idx / fps if fps else 0,
                "class": cls,
                "team": team_label,
                "tracker_id": tid,
                "stable_id": sid,
                "confidence": conf,
                "x1": float(xyxy[0]), "y1": float(xyxy[1]), "x2": float(xyxy[2]), "y2": float(xyxy[3]),
                "cx": float((xyxy[0] + xyxy[2]) / 2), "cy": float((xyxy[1] + xyxy[3]) / 2),
            })
            stats[f"{cls}_detections"] += 1
            if team_label in ["Team A", "Team B"]:
                stats[f"{team_label}_detections"] += 1

        # Hakem çizimi
        for i in range(len(tracked_refs)):
            xyxy = tracked_refs.xyxy[i]
            conf = float(tracked_refs.confidence[i]) if tracked_refs.confidence is not None else None
            tid = int(tracked_refs.tracker_id[i]) if tracked_refs.tracker_id is not None else -1
            sid = referee_sid.update(tid, xyxy, frame_idx)
            draw_detection(annotated, xyxy, "referee", sid, "Referee", conf)
            rows.append({
                "frame": frame_idx,
                "time_sec": frame_idx / fps if fps else 0,
                "class": "referee",
                "team": "Referee",
                "tracker_id": tid,
                "stable_id": sid,
                "confidence": conf,
                "x1": float(xyxy[0]), "y1": float(xyxy[1]), "x2": float(xyxy[2]), "y2": float(xyxy[3]),
                "cx": float((xyxy[0] + xyxy[2]) / 2), "cy": float((xyxy[1] + xyxy[3]) / 2),
            })
            stats["referee_detections"] += 1

        # Top çizimi
        for i in range(len(tracked_balls)):
            xyxy = tracked_balls.xyxy[i]
            conf = float(tracked_balls.confidence[i]) if tracked_balls.confidence is not None else None
            tid = int(tracked_balls.tracker_id[i]) if tracked_balls.tracker_id is not None else -1
            sid = ball_sid.update(tid, xyxy, frame_idx)
            cx = int((xyxy[0] + xyxy[2]) / 2)
            cy = int((xyxy[1] + xyxy[3]) / 2)
            ball_points.append((cx, cy))
            draw_ball_trail(annotated, ball_points)
            draw_detection(annotated, xyxy, "ball", sid, "Ball", conf)
            rows.append({
                "frame": frame_idx,
                "time_sec": frame_idx / fps if fps else 0,
                "class": "ball",
                "team": "Ball",
                "tracker_id": tid,
                "stable_id": sid,
                "confidence": conf,
                "x1": float(xyxy[0]), "y1": float(xyxy[1]), "x2": float(xyxy[2]), "y2": float(xyxy[3]),
                "cx": float((xyxy[0] + xyxy[2]) / 2), "cy": float((xyxy[1] + xyxy[3]) / 2),
            })
            stats["ball_detections"] += 1

        # Üst bilgi paneli
        panel = f"Frame {frame_idx}/{total_to_process} | TeamA:{stats['Team A_detections']} TeamB:{stats['Team B_detections']} Ref:{stats['referee_detections']} Ball:{stats['ball_detections']}"
        cv2.rectangle(annotated, (10, 10), (min(width-10, 1050), 45), (0,0,0), -1)
        cv2.putText(annotated, panel, (20, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2, cv2.LINE_AA)

        writer.write(annotated)

        player_sid.cleanup(frame_idx)
        referee_sid.cleanup(frame_idx)
        ball_sid.cleanup(frame_idx)

        stats["frames"] += 1
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    elapsed = time.time() - start
    summary = {
        "frames_processed": int(stats["frames"]),
        "processing_fps": round(stats["frames"] / elapsed, 2) if elapsed > 0 else 0,
        "thresholds": thresholds,
        "unique_players": int(df[df["class"].isin(["player", "goalkeeper"])] ["stable_id"].nunique()) if len(df) else 0,
        "unique_referees": int(df[df["class"] == "referee"]["stable_id"].nunique()) if len(df) else 0,
        "unique_balls": int(df[df["class"] == "ball"]["stable_id"].nunique()) if len(df) else 0,
        "avg_objects_per_frame": round(len(df) / max(1, stats["frames"]), 2),
        "output_video": out_video,
        "output_csv": out_csv,
    }
    with open(out_summary, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("
İşlem tamamlandı.")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print("Video:", out_video)
    print("CSV:", out_csv)
    print("Summary:", out_summary)
    return df, summary

tracking_df, summary = process_video(VIDEO_PATH, BEST_THRESHOLDS)
display(tracking_df.head())


In [ ]:

# ============================================================
# CELL 12 - Tracking kalite raporu
# ============================================================
def tracking_quality_report(df):
    if len(df) == 0:
        print("CSV boş.")
        return pd.DataFrame()

    rows = []
    for cls, gcls in df.groupby("class"):
        for sid, g in gcls.groupby("stable_id"):
            frames = np.sort(g["frame"].unique())
            if len(frames) == 0:
                continue
            diffs = np.diff(frames)
            gap_count = int((diffs > 1).sum())
            max_gap = int((diffs[diffs > 1] - 1).max()) if gap_count else 0
            rows.append({
                "class": cls,
                "stable_id": sid,
                "n_detections": len(g),
                "start_frame": int(frames.min()),
                "end_frame": int(frames.max()),
                "span": int(frames.max() - frames.min() + 1),
                "coverage": round(len(frames) / max(1, frames.max() - frames.min() + 1), 3),
                "gap_count": gap_count,
                "max_gap": max_gap,
                "mean_conf": round(float(g["confidence"].mean()), 3),
                "team_mode": g["team"].mode().iloc[0] if "team" in g and len(g["team"].mode()) else "",
            })
    rep = pd.DataFrame(rows)
    report_path = os.path.join(OUTPUT_DIR, "tracking_quality_report.csv")
    rep.to_csv(report_path, index=False)

    print("Tracking kalite özeti:")
    if len(rep):
        display(rep.groupby("class").agg(
            unique_ids=("stable_id", "nunique"),
            median_len=("n_detections", "median"),
            mean_len=("n_detections", "mean"),
            short_ids_lt10=("n_detections", lambda s: int((s < 10).sum())),
            mean_coverage=("coverage", "mean"),
            total_gaps=("gap_count", "sum"),
        ).reset_index())
        display(rep.sort_values("n_detections", ascending=False).head(20))

    print("Report CSV:", report_path)
    return rep

quality_df = tracking_quality_report(tracking_df)


In [ ]:

# ============================================================
# CELL 13 - Colab/Kaggle içinde video oynatma ve indirme
# ============================================================
from IPython.display import HTML, display, Video
from base64 import b64encode

video_path = summary["output_video"]
print("Output video:", video_path)

try:
    display(Video(video_path, embed=True, width=960))
except Exception:
    mp4 = open(video_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video width="960" controls><source src="{data_url}" type="video/mp4"></video>'))

# Colab kullanıyorsan indirmek için:
try:
    from google.colab import files
    files.download(summary["output_video"])
    files.download(summary["output_csv"])
except Exception:
    print("Colab files.download kullanılamadı. Kaggle/Local ise OUTPUT_DIR içinden dosyaları al.")



## Ayar önerileri

Takip hâlâ kopuyorsa:
```python
LOST_TRACK_BUFFER = 120
MIN_MATCHING_THRESHOLD = 0.60
PLAYER_STABLE_MAX_DISTANCE = 200
STABLE_MAX_MISSING_FRAMES = 160
```

Çok yanlış ID birleşmesi oluyorsa:
```python
PLAYER_STABLE_MAX_DISTANCE = 100
MIN_MATCHING_THRESHOLD = 0.80
```

Top çok kaçıyorsa:
```python
BALL_THRESHOLDS = [0.02, 0.04, 0.06, 0.08]
BALL_STABLE_MAX_DISTANCE = 260
```

Hakem fazla yanlış yakalanıyorsa:
```python
REFEREE_THRESHOLDS = [0.20, 0.25, 0.30]
```

Oyuncu kaçıyorsa:
```python
PLAYER_THRESHOLDS = [0.08, 0.10, 0.12, 0.15]
GOALKEEPER_THRESHOLDS = [0.08, 0.10, 0.12, 0.15]
```
